In [ ]:
%pip install ipykernel
%pip install opencv-python 
%pip install matplotlib
%pip install matplotlib tk
%pip install plotly nbformat
%pip install scipy
%pip install pyqtgraph PyOpenGL PyQt5
%matplotlib qt


In [ ]:
import threading
import time
from collections import deque
from dataclasses import dataclass

import cv2 as cv
import matplotlib
import numpy as np


def use_window_backend():
    """Notebooks default to an inline backend that closes live figures."""
    try:
        shell = get_ipython()
    except NameError:
        return
    for backend in ("qt", "tk"):
        try:
            shell.run_line_magic("matplotlib", backend)
            return
        except Exception:
            continue


use_window_backend()
import matplotlib.pyplot as plt  # noqa: E402
from matplotlib.cm import ScalarMappable  # noqa: E402
from matplotlib.colors import Normalize  # noqa: E402


FRAME_SHAPE = (384, 256, 2)
FRAME_SIZE = int(np.prod(FRAME_SHAPE))
FONT = cv.FONT_HERSHEY_SIMPLEX


@dataclass
class Config:
    camera_index: int = 0
    left_crop: int = 98              # gray (fallback) mode only
    zoom: int = 4
    rows: int = 80                   # time rows kept in the 3D views
    columns: int = 64                # target number of 3D columns across the image
    update_seconds: float = 0.08     # time between 3D updates
    figure_size: tuple = (18, 8)
    blur_x: float = 0.8              # smoothing across image columns (3D views)
    blur_t: float = 1.0              # smoothing across time rows (3D views)

    # temperature scale (raw mode). Starts at temp_min/max, then follows the scene.
    temp_min: float = 10.0
    temp_max: float = 50.0
    auto_scale: bool = True
    scale_low_pct: float = 1.0
    scale_high_pct: float = 99.8
    min_span: float = 6.0            # never zoom the scale tighter than this many degrees
    scale_smoothing: float = 0.08    # 0..1, lower = steadier scale

    # wisps
    ambient_pct: float = 30.0        # "normal" = this percentile of the recent history
    contrast: float = 0.15           # fraction of the scale span that makes a wisp fully strong
    min_strength: float = 0.15       # weaker cells are not drawn


class Scale:
    """Colour/height range shared by the camera view, the bar and both 3D views."""

    def __init__(self, low, high, unit, auto, cfg):
        self.low, self.high, self.unit, self.auto, self.cfg = low, high, unit, auto, cfg
        self.primed = False

    @property
    def span(self):
        return max(self.high - self.low, 1e-6)

    @property
    def label(self):
        return "°C" if self.unit == "C" else self.unit

    def update(self, values):
        """Follow the scene smoothly; rise fast so hot things never clip."""
        if not self.auto:
            return
        cfg = self.cfg
        lo, hi = np.percentile(values[::2, ::2], (cfg.scale_low_pct, cfg.scale_high_pct))
        hi = max(hi, lo + cfg.min_span)
        if not self.primed:
            self.low, self.high, self.primed = float(lo), float(hi), True
            return
        self.low += cfg.scale_smoothing * (lo - self.low)
        self.high += (0.5 if hi > self.high else cfg.scale_smoothing) * (hi - self.high)


def decode_temps(frame):
    """Return a 192x256 array in °C from a raw TC001 frame."""
    data = frame.reshape(FRAME_SHAPE)[FRAME_SHAPE[0] // 2:]
    raw = data[..., 0].astype(np.uint16) + data[..., 1].astype(np.uint16) * 256
    return (raw / 64.0 - 273.15).astype(np.float32)


def decode_gray(frame, crop):
    """Return the cropped brightness image (0 to 255) from a normal colour frame."""
    return cv.cvtColor(frame[:, crop:], cv.COLOR_BGR2GRAY).astype(np.float32)


def open_raw(index):
    """Return a capture that delivers raw TC001 frames, or None if it doesn't."""
    for name, backend in (("DSHOW", cv.CAP_DSHOW), ("MSMF", cv.CAP_MSMF), ("default", cv.CAP_ANY)):
        capture = cv.VideoCapture(index, backend)
        if not capture.isOpened():
            capture.release()
            continue
        capture.set(cv.CAP_PROP_CONVERT_RGB, 0)
        ok, frame = capture.read()
        print(f"[{name}] read ok={ok}, shape={getattr(frame, 'shape', None)}, "
              f"size={getattr(frame, 'size', None)} (need {FRAME_SIZE})")
        if ok and frame.size == FRAME_SIZE:
            return capture
        capture.release()
    return None


class Camera:
    """Reads frames on a background thread so the display never works from stale, queued frames."""

    def __init__(self, cfg):
        self.crop = cfg.left_crop
        self.capture = open_raw(cfg.camera_index)
        self.raw = self.capture is not None
        if not self.raw:
            self.capture = cv.VideoCapture(cfg.camera_index)
        if not self.capture.isOpened():
            raise RuntimeError("Could not open camera. Try another camera_index.")
        if self.raw:
            self.scale = Scale(cfg.temp_min, cfg.temp_max, "C", cfg.auto_scale, cfg)
        else:
            self.scale = Scale(0, 255, "signal", False, cfg)

        self._frame = None
        self._lock = threading.Lock()
        self._ready = threading.Event()
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._grab, daemon=True)
        self._thread.start()

    def _grab(self):
        while not self._stop.is_set():
            ok, frame = self.capture.read()
            with self._lock:
                self._frame = frame if ok else None
            self._ready.set()
            if not ok:
                break

    def read(self):
        """Return the newest temperatures in °C (raw mode) or brightness, or None."""
        if not self._ready.wait(2.0):
            return None
        self._ready.clear()
        with self._lock:
            frame = self._frame
        if frame is None:
            return None
        return decode_temps(frame) if self.raw else decode_gray(frame, self.crop)

    def release(self):
        self._stop.set()
        self._thread.join(timeout=2.0)
        self.capture.release()


class CameraView:
    """Camera window with a hotspot marker and a temperature bar."""

    def __init__(self, cfg, scale):
        self.zoom = cfg.zoom
        self.scale = scale
        self.bar = None

    def make_bar(self, height):
        gradient = np.linspace(255, 0, height).astype(np.uint8)[:, None].repeat(22, axis=1)
        return cv.applyColorMap(gradient, cv.COLORMAP_INFERNO)

    def show(self, values):
        s = self.scale
        scaled = np.clip((values - s.low) / s.span, 0, 1)
        image = cv.applyColorMap(np.uint8(scaled * 255), cv.COLORMAP_INFERNO)
        image = cv.resize(image, None, fx=self.zoom, fy=self.zoom, interpolation=cv.INTER_CUBIC)
        h, w = image.shape[:2]

        y, x = np.unravel_index(np.argmax(values), values.shape)
        cx, cy = int((x + 0.5) * self.zoom), int((y + 0.5) * self.zoom)
        cv.drawMarker(image, (cx, cy), (255, 255, 255), cv.MARKER_CROSS, 14, 1)
        cv.putText(image, f"{values[y, x]:.1f} {s.unit}", (min(cx + 8, w - 120), max(cy - 8, 16)),
                   FONT, 0.5, (255, 255, 255), 1, cv.LINE_AA)

        if self.bar is None or self.bar.shape[0] != h:
            self.bar = self.make_bar(h)
        labels = np.zeros((h, 80, 3), np.uint8)
        for i in range(6):
            frac = i / 5
            row = int(frac * (h - 1))
            cv.line(labels, (0, row), (5, row), (255, 255, 255), 1)
            cv.putText(labels, f"{s.high - frac * s.span:.1f}", (9, min(max(row + 5, 12), h - 3)),
                       FONT, 0.45, (255, 255, 255), 1, cv.LINE_AA)
        spacer = np.zeros((h, 8, 3), np.uint8)
        cv.imshow("TC001", np.hstack([image, spacer, self.bar, labels]))

    @staticmethod
    def quit_requested():
        return cv.waitKey(1) & 0xFF in (ord("q"), 27)


class History:
    """Rolling rows: hottest value in each block of image columns, averaged over each time step."""

    def __init__(self, width, cfg):
        self.step = max(1, width // cfg.columns)
        self.n = width // self.step
        self.x = np.arange(self.n) * self.step + self.step / 2
        self.rows = deque(maxlen=cfg.rows)
        self.total = np.zeros(self.n, np.float32)
        self.count = 0

    def add_frame(self, values):
        blurred = cv.GaussianBlur(values, (0, 0), 1.0)        # suppress single hot pixels
        h = blurred.shape[0]
        blocks = blurred[:, :self.n * self.step].reshape(h, self.n, self.step)
        self.total += blocks.max(axis=(0, 2))                  # hottest pixel per column block
        self.count += 1

    def commit(self):
        if self.count == 0:
            return
        row = self.total / self.count
        if not self.rows:
            self.rows.extend([row] * self.rows.maxlen)
        else:
            self.rows.append(row)
        self.total = np.zeros(self.n, np.float32)
        self.count = 0

    def as_array(self):
        return np.array(self.rows, np.float32)


def style_dark_axes(ax, xlim, ylim, zlabel):
    ax.set_facecolor("black")
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.set_pane_color((0, 0, 0, 0))
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel("Image X", color="0.7")
    ax.set_ylabel("Time", color="0.7")
    ax.set_zlabel(zlabel, color="0.7")
    ax.tick_params(colors="0.6")
    ax.view_init(45, -70)


class SurfacePanel:
    """Left panel: surface of the hottest value per column over time, in real units."""

    def __init__(self, ax, history, cfg, scale):
        self.ax, self.scale, self.surf = ax, scale, None
        self.x, self.y = np.meshgrid(history.x, np.arange(cfg.rows))
        style_dark_axes(ax, (0, history.x.max()), (0, cfg.rows - 1), f"Signal ({scale.label})")
        ax.set_title("3D map", color="0.85")

    def draw(self, grid):
        s = self.scale
        if self.surf is not None:
            self.surf.remove()
        self.surf = self.ax.plot_surface(
            self.x, self.y, np.clip(grid, s.low, s.high),
            cmap="inferno", vmin=s.low, vmax=s.high,
            linewidth=0, antialiased=False, shade=False,
            rcount=grid.shape[0], ccount=grid.shape[1],
        )
        self.ax.set_zlim(s.low, s.high)


class WispPanel:
    """Right panel: particles for everything hotter than the scene's normal level."""

    PARTICLES = 2

    def __init__(self, ax, history, cfg, scale):
        self.ax, self.cfg, self.scale, self.history = ax, cfg, scale, history
        self.x, self.y = np.meshgrid(history.x, np.arange(cfg.rows))
        rng = np.random.default_rng(7)
        self.jitter = rng.uniform(-0.5, 0.5, (self.PARTICLES, 3, self.x.size))   # fixed, so no flicker
        self.fade = np.linspace(0.15, 1.0, cfg.rows)[:, None]
        self.ambient = None
        self.top = None
        style_dark_axes(ax, (0, history.x.max()), (0, cfg.rows - 1), f"{scale.label} above normal")
        ax.set_title("Wisps", color="0.85")
        self.cloud = ax.scatter([0], [0], [0], s=1, c=[(0, 0, 0, 0)], depthshade=False, linewidths=0)

    def draw(self, grid):
        s, cfg = self.scale, self.cfg
        target = float(np.percentile(grid, cfg.ambient_pct))
        self.ambient = target if self.ambient is None else self.ambient + 0.05 * (target - self.ambient)

        contrast = max(cfg.contrast * s.span, 1e-6)
        excess = np.clip(grid - self.ambient, 0, None)
        strength = np.clip(excess / contrast, 0, 1) * self.fade
        cells = np.flatnonzero(strength.ravel() > cfg.min_strength)

        target_top = max(4 * contrast, float(excess.max()) * 1.1)
        self.top = target_top if self.top is None else self.top + 0.2 * (target_top - self.top)
        self.ax.set_zlim(0, self.top)

        if not cells.size:
            self.cloud._offsets3d = ([0.0], [0.0], [0.0])
            self.cloud.set_facecolor([(0, 0, 0, 0)])
            self.cloud.set_sizes([1])
            return

        heat = excess.ravel()[cells]
        power = strength.ravel()[cells]
        jit = self.jitter[:, :, cells]                                        # (P, 3, n)
        px = (self.x.ravel()[cells] + jit[:, 0] * self.history.step).ravel()
        py = (self.y.ravel()[cells] + jit[:, 1]).ravel()
        pz = np.clip(heat + jit[:, 2] * contrast * 0.15, 0, None).ravel()
        power = np.tile(power, self.PARTICLES)
        temp = self.ambient + np.tile(heat, self.PARTICLES)                    # absolute value -> same colour bar

        colors = plt.get_cmap("inferno")(np.clip((temp - s.low) / s.span, 0, 1))
        colors[:, 3] = np.clip(power * 1.2, 0, 1)
        self.cloud._offsets3d = (px, py, pz)
        self.cloud.set_facecolor(colors)
        self.cloud.set_sizes(4 + 34 * power)


class SideBySide:
    """One window: surface map (left), wisp map (right), shared temperature bar."""

    def __init__(self, history, cfg, scale):
        plt.ion()
        self.history, self.cfg, self.scale = history, cfg, scale
        self.fig = plt.figure("Thermal 3D + wisps", figsize=cfg.figure_size, facecolor="black")
        left = self.fig.add_subplot(121, projection="3d")
        right = self.fig.add_subplot(122, projection="3d")
        self.fig.subplots_adjust(left=0.01, right=0.9, bottom=0.02, top=0.95, wspace=0.02)

        self.mappable = ScalarMappable(norm=Normalize(scale.low, scale.high), cmap="inferno")
        self.cbar = self.fig.colorbar(self.mappable, cax=self.fig.add_axes([0.93, 0.2, 0.014, 0.6]))
        self.cbar.set_label(f"Temperature ({scale.label})", color="0.8")
        self.cbar.ax.tick_params(colors="0.8")
        self.cbar.outline.set_edgecolor("0.5")

        self.panels = [
            SurfacePanel(left, history, cfg, scale),
            WispPanel(right, history, cfg, scale),
        ]

    def draw(self):
        """Redraw both panels. Returns False once the window is closed."""
        grid = cv.GaussianBlur(self.history.as_array(), (0, 0),
                               sigmaX=self.cfg.blur_x, sigmaY=self.cfg.blur_t)
        self.mappable.set_clim(self.scale.low, self.scale.high)
        self.cbar.update_normal(self.mappable)
        for panel in self.panels:
            panel.draw(grid)
        plt.pause(0.001)
        return plt.fignum_exists(self.fig.number)


def run(cfg=Config()):
    if "inline" in matplotlib.get_backend().lower():
        raise RuntimeError("Inline backend closes the figure. Run %matplotlib qt, then restart the kernel.")

    camera = Camera(cfg)
    print("Reading temperatures (°C)." if camera.raw else "Reading brightness only: this camera sends processed frames.")
    view = CameraView(cfg, camera.scale)
    history = maps = None
    last_update = 0.0

    try:
        while True:
            values = camera.read()
            if values is None:
                break
            camera.scale.update(values)

            if history is None:
                history = History(values.shape[1], cfg)
                maps = SideBySide(history, cfg, camera.scale)

            view.show(values)
            history.add_frame(values)

            if time.monotonic() - last_update >= cfg.update_seconds:
                last_update = time.monotonic()
                history.commit()
                if not maps.draw():
                    break

            if view.quit_requested():
                break
    finally:
        camera.release()
        cv.destroyAllWindows()
        plt.close("all")


if __name__ == "__main__":
    run()
    

In [2]:
#simulated - one window: live map with hotspot boxes, tracked-only view, 3D views and data panels SIMULATED
# Drawn with pyqtgraph + OpenGL. Run the install cell first, and restart the kernel before re-running.
#
#   1. Setup              4. Hotspots (detect + track)     7. run()  <- start here
#   2. Settings           5. 3D views
#   3. Frame sources      6. The window
import ctypes
import threading
import time
from collections import deque
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from itertools import takewhile

import cv2 as cv
import numpy as np
import OpenGL
OpenGL.ERROR_CHECKING = False            # PyOpenGL checks every GL call by default: slow
import pyqtgraph as pg                   # noqa: E402
import pyqtgraph.opengl as gl            # noqa: E402
from pyqtgraph.Qt import QtCore, QtGui, QtWidgets   # noqa: E402

Qt = QtCore.Qt
QShortcut = getattr(QtGui, "QShortcut", None) or QtWidgets.QShortcut   # Qt6 / Qt5

# The OpenGL views must not each wait for the screen refresh: FramePacer waits once per frame.
# (Only takes effect before the first OpenGL view exists - hence "restart the kernel".)
_format = QtGui.QSurfaceFormat.defaultFormat()
_format.setSwapInterval(0)
QtGui.QSurfaceFormat.setDefaultFormat(_format)


# ==============================================================================================
# 2. Settings
# ==============================================================================================

@dataclass
class Config:
    simulate: bool = True             # True = synthetic scene, False = real TC001 camera
    sim_fps: float = 60.0             # target rate; snapped to the screen (144 Hz -> 72 fps)
    camera_index: int = 0
    left_crop: int = 98               # gray (fallback) camera mode only
    window_size: tuple = (1800, 1000)

    # 3D views
    rows: int = 80                    # time rows kept
    columns: int = 64                 # columns across the image
    update_seconds: float = 0.08      # time per row
    blur_x: float = 0.8               # smoothing across columns
    blur_t: float = 1.0               # smoothing across time
    panel_seconds: float = 0.2        # list / data / chart refresh interval

    # colour scale: starts at temp_min..temp_max, then follows the scene
    temp_min: float = 10.0
    temp_max: float = 50.0
    auto_scale: bool = True
    scale_low_pct: float = 1.0
    scale_high_pct: float = 99.8
    min_span: float = 6.0             # never tighter than this many degrees
    scale_smoothing: float = 0.08     # 0..1, lower = steadier

    # wisps
    ambient_pct: float = 30.0         # "normal" = this percentile of the history
    contrast: float = 0.15            # fraction of the scale span that makes a wisp fully strong
    min_strength: float = 0.15        # weaker cells are not drawn

    # hotspot detection
    hot_delta: float = 4.0            # hotspot = this much above the scene median...
    hot_fraction: float = 0.12        # ...or this fraction of the scale span, whichever is larger
    prominence: float = 5.0           # a peak needs this rise above the dip to a hotter neighbour
    min_area: int = 20                # pixels
    min_rise: float = 1.5             # peak must be this far above the hotspot level
    detect_step: int = 2              # detect on an image this many times smaller (speed)

    # hotspot tracking
    match_distance: float = 30.0      # pixels a hotspot may jump and keep its ID...
    max_speed: float = 80.0           # ...plus this many pixels per second since last seen
    confirm_frames: int = 3           # frames before a hotspot gets an ID
    forget_frames: int = 20           # frames a hotspot may be missing before it is dropped
    focus_margin: int = 4             # pixels kept around the picked hotspot
    chart_seconds: float = 30.0       # length of the temperature chart
    log_length: int = 30000           # samples kept per hotspot (chart + CSV)


# ==============================================================================================
# 3. Frame sources: real camera, simulated scene, and the frame pacer
# ==============================================================================================

FRAME_SHAPE = (384, 256, 2)           # raw TC001 frame: image half + temperature half
FRAME_SIZE = int(np.prod(FRAME_SHAPE))


class Scale:
    """Colour/height range shared by every view. Follows the scene; rises fast so nothing clips."""

    def __init__(self, low, high, unit, auto, cfg):
        self.low, self.high, self.auto, self.cfg = low, high, auto, cfg
        self.label = "°C" if unit == "C" else unit
        self.primed = False

    @property
    def span(self):
        return max(self.high - self.low, 1e-6)

    def update(self, values):
        if not self.auto:
            return
        cfg = self.cfg
        lo, hi = np.percentile(values[::2, ::2], (cfg.scale_low_pct, cfg.scale_high_pct))
        hi = max(hi, lo + cfg.min_span)
        if not self.primed:
            self.low, self.high, self.primed = float(lo), float(hi), True
            return
        self.low += cfg.scale_smoothing * (lo - self.low)
        self.high += (0.5 if hi > self.high else cfg.scale_smoothing) * (hi - self.high)


def open_raw(index):
    """A capture that delivers raw TC001 frames, or None."""
    for name, backend in (("DSHOW", cv.CAP_DSHOW), ("MSMF", cv.CAP_MSMF), ("default", cv.CAP_ANY)):
        capture = cv.VideoCapture(index, backend)
        if capture.isOpened():
            capture.set(cv.CAP_PROP_CONVERT_RGB, 0)
            ok, frame = capture.read()
            print(f"[{name}] read ok={ok}, size={getattr(frame, 'size', None)} (need {FRAME_SIZE})")
            if ok and frame.size == FRAME_SIZE:
                return capture
        capture.release()
    return None


class Camera:
    """TC001 camera. A background thread keeps only the newest frame, so nothing queues up."""

    def __init__(self, cfg):
        self.crop = cfg.left_crop
        self.capture = open_raw(cfg.camera_index)
        self.raw = self.capture is not None
        if not self.raw:
            self.capture = cv.VideoCapture(cfg.camera_index)
        if not self.capture.isOpened():
            raise RuntimeError("Could not open camera. Try another camera_index.")
        self.scale = (Scale(cfg.temp_min, cfg.temp_max, "C", cfg.auto_scale, cfg) if self.raw
                      else Scale(0, 255, "signal", False, cfg))
        self._frame, self._lock = None, threading.Lock()
        self._ready, self._stop = threading.Event(), threading.Event()
        self._thread = threading.Thread(target=self._grab, daemon=True)
        self._thread.start()

    def _grab(self):
        while not self._stop.is_set():
            ok, frame = self.capture.read()
            with self._lock:
                self._frame = frame if ok else None
            self._ready.set()
            if not ok:
                break

    def read(self):
        """Newest frame in °C (raw mode) or brightness 0..255; None if the camera stopped."""
        if not self._ready.wait(2.0):
            return None
        self._ready.clear()
        with self._lock:
            frame = self._frame
        if frame is None:
            return None
        if not self.raw:
            return cv.cvtColor(frame[:, self.crop:], cv.COLOR_BGR2GRAY).astype(np.float32)
        data = frame.reshape(FRAME_SHAPE)[FRAME_SHAPE[0] // 2:]
        raw = data[..., 0].astype(np.uint16) + data[..., 1].astype(np.uint16) * 256
        return (raw / 64.0 - 273.15).astype(np.float32)

    def release(self):
        self._stop.set()
        self._thread.join(timeout=2.0)
        self.capture.release()


class SimulatedCamera:
    """Stand-in for Camera with synthetic 192x256 °C frames, based on the clock (any frame rate).

    A cool room with a warm wall on the right, a person walking side to side, a pulsing hot spot
    (like a soldering iron) and a warm hand sweeping across the top every 9 s, plus sensor noise.
    """

    H, W = 192, 256

    def __init__(self, cfg):
        self.raw = True
        self.scale = Scale(cfg.temp_min, cfg.temp_max, "C", cfg.auto_scale, cfg)
        self.rng = np.random.default_rng(1)
        self.gx, self.gy = np.arange(self.W, dtype=np.float32), np.arange(self.H, dtype=np.float32)
        texture = cv.resize(self.rng.normal(size=(12, 16)).astype(np.float32), (self.W, self.H),
                            interpolation=cv.INTER_CUBIC)
        wall = 3.0 * np.clip((self.gx - 150) / 106, 0, 1) ** 2
        self.room = 21.0 + texture + wall[None, :]
        self.t0 = time.monotonic()

    def _blob(self, cx, cy, sx, sy, amp):
        bx = np.exp(-((self.gx - cx) ** 2) / (2 * sx * sx))
        by = np.exp(-((self.gy - cy) ** 2) / (2 * sy * sy))
        return amp * by[:, None] * bx[None, :]

    def read(self):
        t = time.monotonic() - self.t0
        px, py = self.W / 2 + 70 * np.sin(0.35 * t), self.H * 0.62 + 6 * np.sin(1.1 * t)
        sweep = (t % 9.0) / 9.0
        frame = (self.room
                 + self._blob(px, py, 22, 48, 11.0)                                  # torso
                 + self._blob(px, py - 58, 10, 12, 13.0)                             # head
                 + self._blob(238, 150, 5, 5, 24 + 8 * np.sin(1.7 * t))              # pulsing spot
                 + self._blob(-20 + sweep * (self.W + 40), 50, 8, 8, 14 * np.sin(np.pi * sweep))  # hand
                 + self.rng.normal(0, 0.08, self.room.shape))
        return frame.astype(np.float32)

    def release(self):
        pass


class FramePacer:
    """Times frames to the screen's refresh so every frame stays up equally long.

    A plain 60 fps timer on a 144 Hz screen shows frames for 2, 3, 2, 3... refreshes, which looks
    like stutter. Instead we wait for real refreshes (Windows DwmFlush) and show each frame for a
    whole number of them: 60 Hz -> 60 fps, 144 Hz -> 72 fps, 120 Hz -> 60 fps.

    If this computer can't keep up (over 15% of frames late), it steps down to the next even
    rhythm (e.g. 72 -> 48 fps): a steady lower rate looks far smoother than one that stutters.
    """

    MAX_LATE = 0.15

    def __init__(self, fps):
        self.period, self.refresh, self.flush = 1.0 / fps, None, None
        try:
            flush = ctypes.windll.dwmapi.DwmFlush
            stamps = []
            for _ in range(12):
                if flush() != 0:
                    raise OSError
                stamps.append(time.perf_counter())
            refresh = float(np.median(np.diff(stamps)))
            if 0.002 < refresh < 0.05:
                self.flush, self.refresh = flush, refresh
                self.period = max(1, int(self.period / refresh + 0.05)) * refresh
        except (AttributeError, OSError):
            pass                                        # not Windows: fall back to a timer
        self.next = self.started = time.perf_counter()
        self.frames = self.late = 0
        self.recent = deque(maxlen=120)                 # late (True) / on time, last 120 frames

    @property
    def fps(self):
        return 1.0 / self.period

    def wait(self):
        """Return just after the refresh on which the next frame should appear."""
        if self.flush is None:
            late = time.perf_counter() > self.next + 0.25 * self.period
            time.sleep(max(0.0, self.next - time.perf_counter()))
            self.next = max(self.next + self.period, time.perf_counter())
        else:
            while True:
                self.flush()                            # blocks until the next refresh
                now = time.perf_counter()
                if now >= self.next - 0.5 * self.refresh:
                    break
            late = now > self.next + 0.5 * self.refresh     # missed its refresh
            self.next = now + self.period
        self._count(late)

    def _count(self, late):
        self.frames += 1
        self.late += late
        self.recent.append(late)
        too_slow = len(self.recent) == self.recent.maxlen and sum(self.recent) > self.MAX_LATE * len(self.recent)
        if too_slow and self.refresh and self.fps > 25:
            self.period += self.refresh
            self.recent.clear()
            print(f"This computer can't keep up: switching to a steady {self.fps:.0f} fps.")

    def report(self):
        seconds = time.perf_counter() - self.started
        on_time = 100 * (1 - self.late / max(self.frames, 1))
        return (f"Ran {self.frames} frames in {seconds:.0f} s ({self.frames / max(seconds, 1e-9):.0f} fps), "
                f"{on_time:.0f}% on time. Final rate {self.fps:.0f} fps.")


# ==============================================================================================
# 4. Hotspots: find them in each frame, then follow them over time
# ==============================================================================================

@dataclass
class Detection:
    box: tuple          # x, y, w, h in image pixels
    center: tuple       # heat-weighted centre (x, y)
    peak: float         # hottest reading
    peak_xy: tuple      # where that reading is
    mean: float         # average reading
    area: int           # pixels
    mask: np.ndarray    # bool image, True inside the hotspot


def segment_peaks(img, threshold, prominence):
    """Label pixels above `threshold`, one label per peak that clearly stands on its own.

    Pixels join a touching neighbour's region, hottest first. When two regions meet, the cooler
    one is merged unless its peak rises `prominence` above the meeting point - so a head on a
    body stays one hotspot, while a person next to a hot mug gives two.
    """
    h, w = img.shape
    pw = w + 2
    padded = np.full((h + 2, pw), -np.inf, np.float32)
    padded[1:-1, 1:-1] = img
    flat = padded.ravel()
    idx = np.flatnonzero(flat > threshold)
    if idx.size == 0:
        return np.zeros((h, w), np.int32)

    order = idx[np.argsort(-flat[idx], kind="stable")]
    neighbours = (-pw - 1, -pw, -pw + 1, -1, 1, pw - 1, pw, pw + 1)
    owner, parent, peak = {}, [], []          # pixel -> region, region tree, region peak

    def root(r):
        while parent[r] != r:
            parent[r] = parent[parent[r]]
            r = parent[r]
        return r

    for p, v in zip(order.tolist(), flat[order].tolist()):
        roots = []
        for o in neighbours:
            r = owner.get(p + o)
            if r is not None and (r := root(r)) not in roots:
                roots.append(r)
        if not roots:                             # a new peak
            owner[p] = len(parent)
            parent.append(len(parent))
            peak.append(v)
            continue
        roots.sort(key=peak.__getitem__, reverse=True)
        for r in roots[1:]:
            if peak[r] - v < prominence:          # not prominent enough: merge into the hotter one
                parent[r] = roots[0]
        owner[p] = roots[0]

    names = {}
    pixels = np.fromiter(owner.keys(), np.int64, len(owner))
    labels = np.fromiter((names.setdefault(root(r), len(names) + 1) for r in owner.values()),
                         np.int32, len(owner))
    out = np.zeros(flat.size, np.int32)
    out[pixels] = labels
    return out.reshape(h + 2, pw)[1:-1, 1:-1].copy()


class HotspotDetector:
    """Finds the regions clearly hotter than the scene and measures each one."""

    def __init__(self, cfg):
        self.cfg = cfg
        self.ambient = self.threshold = None      # smoothed scene median / current hotspot level

    def detect(self, values, scale):
        cfg = self.cfg
        blurred = cv.GaussianBlur(values, (0, 0), 1.2)
        median = float(np.median(blurred[::2, ::2]))
        self.ambient = median if self.ambient is None else self.ambient + 0.1 * (median - self.ambient)
        self.threshold = self.ambient + max(cfg.hot_delta, cfg.hot_fraction * scale.span)

        # Segment a smaller copy (much faster), then scale the labels back up.
        h, w = values.shape
        f = max(1, int(cfg.detect_step))
        small = cv.resize(blurred, (w // f, h // f), interpolation=cv.INTER_AREA) if f > 1 else blurred
        small_labels = segment_peaks(small, self.threshold, max(cfg.prominence, 0.08 * scale.span))
        labels = np.zeros((h, w), np.int32)
        up = small_labels.repeat(f, axis=0).repeat(f, axis=1)
        labels[:up.shape[0], :up.shape[1]] = up
        labels[blurred <= self.threshold] = 0

        found = []
        for label in range(1, int(small_labels.max()) + 1):
            mask = labels == label
            ys, xs = np.nonzero(mask)
            if xs.size < cfg.min_area or blurred[ys, xs].max() < self.threshold + cfg.min_rise:
                continue                          # too small, or barely warm (texture, edges)
            weight = np.clip(blurred[ys, xs] - self.ambient, 1e-3, None)
            readings = values[ys, xs]
            k = int(np.argmax(readings))
            x0, y0 = int(xs.min()), int(ys.min())
            found.append(Detection(
                box=(x0, y0, int(xs.max()) - x0 + 1, int(ys.max()) - y0 + 1),
                center=(float(xs @ weight / weight.sum()), float(ys @ weight / weight.sum())),
                peak=float(readings[k]), peak_xy=(int(xs[k]), int(ys[k])),
                mean=float(readings.mean()), area=int(xs.size), mask=mask))
        return found


class Track:
    """One hotspot followed over time, with everything recorded about it."""

    def __init__(self, det, now, cfg):
        self.id = None                                  # given once the hotspot is confirmed
        self.center = self.start = self.pos = det.center    # smoothed / first / raw centre
        self.vel = (0.0, 0.0)                           # pixels per second, smoothed
        self.first_seen = self.last_seen = now
        self.hits = self.missed = 0
        self.distance = self.total = 0.0
        self.high, self.low = -np.inf, np.inf
        self.log = deque(maxlen=cfg.log_length)         # (t, peak, mean, cx, cy, bx, by, bw, bh)
        self.see(det, now)

    def see(self, det, now):
        if self.hits:
            dt = now - self.last_seen
            if dt > 0:
                vx, vy = (det.center[0] - self.pos[0]) / dt, (det.center[1] - self.pos[1]) / dt
                self.vel = (self.vel[0] + 0.4 * (vx - self.vel[0]), self.vel[1] + 0.4 * (vy - self.vel[1]))
            cx, cy = self.center
            nx, ny = (cx + det.center[0]) / 2, (cy + det.center[1]) / 2
            self.distance += float(np.hypot(nx - cx, ny - cy))
            self.center, self.pos = (nx, ny), det.center
        self.det, self.last_seen = det, now
        self.hits, self.missed = self.hits + 1, 0
        self.high, self.low = max(self.high, det.peak), min(self.low, det.peak)
        self.total += det.peak
        self.log.append((now, det.peak, det.mean, *self.center, *det.box))

    def predict(self, now):
        dt = min(now - self.last_seen, 1.0)
        return self.pos[0] + self.vel[0] * dt, self.pos[1] + self.vel[1] * dt

    @property
    def average(self):
        return self.total / max(self.hits, 1)

    def recent(self, since):
        """Log entries newer than `since`, oldest first."""
        return list(takewhile(lambda e: e[0] >= since, reversed(self.log)))[::-1]

    def speed(self, window=1.0):
        """Pixels per second over the last `window` seconds."""
        span = self.recent(self.log[-1][0] - window) if self.log else []
        if len(span) < 2 or span[-1][0] <= span[0][0]:
            return 0.0
        a, b = span[0], span[-1]
        return float(np.hypot(b[3] - a[3], b[4] - a[4]) / (b[0] - a[0]))


class Tracker:
    """Keeps hotspot IDs stable by matching each detection to the nearest predicted track."""

    def __init__(self, cfg):
        self.cfg, self.tracks, self.next_id = cfg, [], 1

    def update(self, detections, now, keep=None):
        cfg = self.cfg
        pairs = []                                      # (distance, track, detection) within reach
        for i, t in enumerate(self.tracks):
            px, py = t.predict(now)
            reach = cfg.match_distance + cfg.max_speed * min(now - t.last_seen, 0.75)
            for j, d in enumerate(detections):
                dist = float(np.hypot(px - d.center[0], py - d.center[1]))
                if dist <= reach:
                    pairs.append((dist, i, j))
        pairs.sort()
        used_t, used_d = set(), set()
        for _, i, j in pairs:                           # closest pairs first
            if i not in used_t and j not in used_d:
                used_t.add(i)
                used_d.add(j)
                self.tracks[i].see(detections[j], now)

        survivors = []
        for i, t in enumerate(self.tracks):
            if i not in used_t:
                t.missed += 1
                limit = cfg.forget_frames if t.id is not None else 1
                if t.missed > limit and (keep is None or t.id != keep):
                    continue                            # gone (the picked one is always kept)
            if t.id is None and t.hits >= cfg.confirm_frames:
                t.id, self.next_id = self.next_id, self.next_id + 1
            survivors.append(t)
        self.tracks = survivors + [Track(d, now, cfg) for j, d in enumerate(detections) if j not in used_d]

    def get(self, tid):
        return next((t for t in self.tracks if t.id == tid), None) if tid is not None else None

    def listed(self):
        """Confirmed hotspots, by ID."""
        return sorted((t for t in self.tracks if t.id is not None), key=lambda t: t.id)


# ==============================================================================================
# 5. 3D views: a rolling history of the hottest value per column, shown two ways
# ==============================================================================================

class History:
    """Rows of "hottest value per column block", one row per update_seconds, newest last."""

    def __init__(self, width, cfg):
        self.step = max(1, width // cfg.columns)
        self.n = width // self.step
        self.x = np.arange(self.n) * self.step + self.step / 2
        self.rows = deque(maxlen=cfg.rows)
        self.serial = 0                                 # rows committed so far (never reset)
        self.reset()

    def reset(self):
        self.rows.clear()
        self.total, self.count = np.zeros(self.n, np.float32), 0

    def add_frame(self, values):
        blurred = cv.GaussianBlur(values, (0, 0), 1.0)                     # ignore single hot pixels
        blocks = blurred[:, :self.n * self.step].reshape(blurred.shape[0], self.n, self.step)
        self.total += blocks.max(axis=(0, 2))
        self.count += 1

    def commit(self):
        if self.count:
            row = self.total / self.count
            self.rows.extend([row] * (self.rows.maxlen if not self.rows else 1))   # first row fills all
            self.total, self.count = np.zeros(self.n, np.float32), 0
            self.serial += 1

    def with_live_row(self):
        """Committed rows plus the row still being filled."""
        live = self.total / self.count if self.count else self.rows[-1]
        return np.vstack([np.asarray(self.rows, np.float32), live[None]])


BG, PANEL, EDGE, TEXT, MUTED = "#0c0e13", "#151821", "#2b303b", "#e3e6ec", "#8b92a3"
SPOT_COLORS = ("#38d1ff", "#7dff6a", "#ff6bd6", "#f4f4f4", "#b69bff", "#00f5c4", "#6ba4ff", "#d4ff3a")
INFERNO = pg.colormap.get("inferno")
LUT_BGRA = np.c_[INFERNO.getLookupTable(0.0, 1.0, 256)[:, ::-1], np.full(256, 255, np.uint8)]  # 2D images
LUT_RGBA = INFERNO.getLookupTable(0.0, 1.0, 256, alpha=True).astype(np.float32) / 255           # 3D colours


def spot_color(tid):
    return SPOT_COLORS[(tid - 1) % len(SPOT_COLORS)]


def colour_index(values, low, span):
    """Readings -> 0..255 position on the colour scale."""
    return np.clip((values - low) * (255.0 / span), 0, 255).astype(np.uint8)


class View3D:
    """An OpenGL 3D view: image X across, time going back, value up. Drag to rotate, wheel to zoom.

    Rows glide: each frame the history slides by the fraction of a row that has passed, and the
    row still filling is drawn at the edge, so the view never jumps.
    """

    SX, SY, SZ = 256.0, 200.0, 110.0                   # size of the drawing box in GL units

    def __init__(self, history, cfg, scale, height_label):
        self.cfg, self.scale, self.height_label, self.range_text = cfg, scale, height_label, ""
        self.widget = gl.GLViewWidget()
        self.widget.setBackgroundColor(BG)
        self.widget.setMinimumHeight(200)
        self.xs = (history.x / history.x.max() * self.SX).astype(np.float32)
        self.dy = self.SY / cfg.rows
        self.ys = (np.arange(cfg.rows + 1) * self.dy).astype(np.float32)     # rows + the live one
        grid = gl.GLGridItem()
        grid.setSize(self.SX, self.SY)
        grid.setSpacing(self.SX / 8, self.SY / 8)
        grid.translate(self.SX / 2, self.SY / 2, 0)
        grid.setColor((255, 255, 255, 35))
        self.widget.addItem(grid)
        self.widget.setCameraPosition(pos=QtGui.QVector3D(self.SX / 2, self.SY / 2, self.SZ * 0.3),
                                      distance=360, elevation=34, azimuth=-112)

    def place(self, item, frac, low, high):
        item.resetTransform()
        item.translate(0, -frac * self.dy, 0)
        self.range_text = f"height: {self.height_label}  {low:.1f} to {high:.1f}"


class Surface3D(View3D):
    """Surface of the hottest value per column over time."""

    def __init__(self, history, cfg, scale):
        super().__init__(history, cfg, scale, f"Signal ({scale.label})")
        self.surf = gl.GLSurfacePlotItem(x=self.xs, y=self.ys, z=np.zeros((self.xs.size, self.ys.size)),
                                         computeNormals=False, smooth=False)
        self.widget.addItem(self.surf)

    def draw(self, grid, frac, serial, ease):
        s = self.scale
        level = colour_index(grid.T, s.low, s.span)                         # (columns, rows)
        self.surf.setData(z=level * (self.SZ / 255), colors=LUT_RGBA[level].reshape(-1, 4))
        self.place(self.surf, frac, s.low, s.high)


class Wisps3D(View3D):
    """Glowing particles for everything hotter than the scene's normal level."""

    PARTICLES = 2

    def __init__(self, history, cfg, scale):
        super().__init__(history, cfg, scale, f"{scale.label} above normal")
        self.gx, self.gy = (a.ravel() for a in np.meshgrid(self.xs, self.ys))
        # Fixed random offsets per (row, column), tied to the row so particles move with it.
        rng = np.random.default_rng(7)
        self.jitter = rng.uniform(-0.5, 0.5, (self.PARTICLES, 3, self.ys.size, self.xs.size)).astype(np.float32)
        self.dx = self.xs[1] - self.xs[0]
        self.ambient = self.top = None
        self.cloud = gl.GLScatterPlotItem(pos=np.zeros((1, 3)), color=(0, 0, 0, 0), size=1, pxMode=True)
        self.cloud.setGLOptions("additive")
        self.widget.addItem(self.cloud)

    def draw(self, grid, frac, serial, ease):
        s, cfg, rows = self.scale, self.cfg, grid.shape[0]
        normal = float(np.percentile(grid, cfg.ambient_pct))
        self.ambient = normal if self.ambient is None else self.ambient + (1 - 0.95 ** ease) * (normal - self.ambient)
        contrast = max(cfg.contrast * s.span, 1e-6)
        excess = np.clip(grid - self.ambient, 0, None)
        age_fade = 0.15 + 0.85 * np.clip((np.arange(rows) - frac) / (rows - 2), 0, 1)[:, None]
        strength = np.clip(excess / contrast, 0, 1) * age_fade
        top = max(4 * contrast, float(excess.max()) * 1.1)
        self.top = top if self.top is None else self.top + (1 - 0.8 ** ease) * (top - self.top)
        self.place(self.cloud, frac, 0.0, self.top)

        cells = np.flatnonzero(strength.ravel() > cfg.min_strength)
        if not cells.size:
            self.cloud.setData(pos=np.zeros((1, 3)), color=(0, 0, 0, 0), size=1)
            return
        row_order = (serial - (rows - 1) + np.arange(rows)) % rows
        jit = self.jitter[:, :, row_order].reshape(self.PARTICLES, 3, -1)[:, :, cells]   # (P, 3, n)
        heat, power = excess.ravel()[cells], strength.ravel()[cells]
        pos = np.empty((self.PARTICLES * cells.size, 3), np.float32)
        pos[:, 0] = (self.gx[cells] + jit[:, 0] * self.dx).ravel()
        pos[:, 1] = (self.gy[cells] + jit[:, 1] * self.dy).ravel()
        pos[:, 2] = (np.clip(heat + jit[:, 2] * contrast * 0.15, 0, None) * (self.SZ / self.top)).ravel()
        power = np.tile(power, self.PARTICLES)
        colors = LUT_RGBA[colour_index(self.ambient + np.tile(heat, self.PARTICLES), s.low, s.span)]
        colors[:, 3] = power * 0.5                                            # soft additive glow
        self.cloud.setData(pos=pos, color=colors, size=3 + 9 * power)


# ==============================================================================================
# 6. The window
# ==============================================================================================

STYLE = f"""
QWidget#root {{ background: {BG}; }}
QLabel {{ color: {TEXT}; }}
QLabel[muted="true"] {{ color: {MUTED}; }}
QFrame[panel="true"] {{ background: {PANEL}; border: 1px solid {EDGE}; border-radius: 4px; }}
QListWidget {{ background: {PANEL}; border: 1px solid {EDGE}; color: {TEXT}; outline: 0; }}
QListWidget::item {{ padding: 2px 4px; }}
QPushButton {{ background: {PANEL}; color: {TEXT}; border: 1px solid {EDGE}; padding: 6px 10px; border-radius: 3px; }}
QPushButton:hover {{ background: #232838; }}
"""


def make_label(text="", muted=False, size=9, bold=False):
    w = QtWidgets.QLabel(text)
    w.setTextFormat(Qt.TextFormat.PlainText)            # plain text paints much faster than rich text
    w.setProperty("muted", muted)
    font = w.font()
    font.setPointSizeF(size)
    font.setBold(bold)
    w.setFont(font)
    return w


def set_text(widget, text, setter=None):
    """Change text only when it differs: every change makes Qt lay the window out again."""
    if getattr(widget, "_shown", None) != text:
        widget._shown = text
        (setter or widget.setText)(text)


class FastImage(QtWidgets.QGraphicsItem):
    """Shows a BGRA image. That is Qt's native pixel format, so painting is a plain copy."""

    def __init__(self, w, h):
        super().__init__()
        self.rect = QtCore.QRectF(-0.5, -0.5, w, h)
        self.pixels = self.qimage = None

    def boundingRect(self):
        return self.rect

    def set_pixels(self, bgra):
        self.pixels = np.ascontiguousarray(bgra)          # Qt reads straight from this buffer
        self.qimage = pg.functions.ndarray_to_qimage(self.pixels, QtGui.QImage.Format.Format_ARGB32_Premultiplied)
        self.update()

    def paint(self, painter, *args):
        if self.qimage is not None:
            painter.drawImage(self.rect, self.qimage)


class Window(QtWidgets.QWidget):
    closed = False

    def closeEvent(self, event):
        self.closed = True
        event.accept()


class ThermalApp:
    """Live map with every hotspot boxed, a tracked-only view, both 3D views and data panels.

    Pick a hotspot by clicking its box or its list row: the live map keeps showing everything,
    while the tracked view, 3D views, data panel and chart follow only that hotspot.
    """

    ROWS = 8                                            # hotspots shown in the list

    def __init__(self, cfg, scale, history, tracker, detector, shape, simulated):
        pg.setConfigOptions(antialias=False)
        self.qapp = pg.mkQApp()
        self.cfg, self.scale, self.history, self.tracker, self.detector = cfg, scale, history, tracker, detector
        self.height, self.width = shape
        self.selected = None                            # picked hotspot ID (None = all)
        self.message, self.message_until = "", 0.0
        self.row_ids, self.drawn = [], []
        self.frames, self.fps, self.fps_t0 = 0, 0.0, time.monotonic()
        self.soft_key = self.soft = None                # cached soft mask of the picked hotspot
        self.last_3d, self.last_panels, self.panel_step = time.monotonic(), -np.inf, 0
        m = max(0, int(cfg.focus_margin))
        self.kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (2 * m + 1, 2 * m + 1))

        win = self.window = Window()
        win.setObjectName("root")
        win.setWindowTitle("Thermal hotspot tracker")
        win.setStyleSheet(STYLE)
        win.resize(*cfg.window_size)
        outer = QtWidgets.QVBoxLayout(win)
        outer.setContentsMargins(12, 8, 12, 10)
        outer.addWidget(make_label("Thermal hotspot tracker" + ("   ·   SIMULATION" if simulated else ""),
                                   size=14, bold=True))
        outer.addWidget(make_label("Click a box (or a list row) to track one hotspot   ·   N / P  next / previous"
                                   "   ·   A  show all   ·   1-8  pick from list   ·   S  save CSV   ·   Q  quit"
                                   "   ·   drag the 3D views to rotate them", muted=True))
        body = QtWidgets.QHBoxLayout()
        left, right = QtWidgets.QVBoxLayout(), QtWidgets.QVBoxLayout()
        body.addLayout(left, 7)
        body.addLayout(right, 3)
        outer.addLayout(body, 1)
        self._build_maps(left)
        self._build_3d(left)
        self._build_side(right)
        self._bind_keys()
        win.show()
        self.pump()

    # ---- building the window ----

    def _build_maps(self, layout):
        """Live map and tracked-only view, side by side, sharing one colour bar."""
        self.maps = pg.GraphicsLayoutWidget()
        self.maps.setBackground(BG)
        self.live_plot, self.live_img, self.live_peak = self._map_plot()
        self.focus_plot, self.focus_img, self.focus_peak = self._map_plot()
        self.cbar = pg.ColorBarItem(values=(self.scale.low, self.scale.high), colorMap=INFERNO,
                                    interactive=False, width=12, label=self.scale.label)
        self.maps.addItem(self.cbar)
        self.live_boxes, self.focus_boxes = [], []
        self.focus_note = pg.TextItem("", color=TEXT, anchor=(0.5, 0.5), fill=pg.mkBrush(BG), border=pg.mkPen(EDGE))
        self.focus_note.setPos(self.width / 2, self.height / 2)
        self.focus_plot.addItem(self.focus_note)
        self.maps.scene().sigMouseClicked.connect(self._on_map_click)
        layout.addWidget(self.maps, 11)

    def _map_plot(self):
        p = self.maps.addPlot()
        p.hideAxis("left")
        p.hideAxis("bottom")
        p.setMenuEnabled(False)
        p.hideButtons()
        vb = p.getViewBox()
        vb.setMouseEnabled(False, False)
        vb.invertY(True)
        vb.setAspectLocked(True)
        vb.setRange(xRange=(-0.5, self.width - 0.5), yRange=(-0.5, self.height - 0.5), padding=0)
        vb.setBorder(pg.mkPen(EDGE))
        image = FastImage(self.width, self.height)
        p.addItem(image)
        peak = pg.ScatterPlotItem(symbol="+", size=16, pen=pg.mkPen("w", width=1.6), brush=None)
        peak.setZValue(10)
        p.addItem(peak)
        return p, image, peak

    def _build_3d(self, layout):
        self.views = [Surface3D(self.history, self.cfg, self.scale), Wisps3D(self.history, self.cfg, self.scale)]
        self.view_titles = [make_label(size=10) for _ in self.views]
        row = QtWidgets.QHBoxLayout()
        for view, title in zip(self.views, self.view_titles):
            col = QtWidgets.QVBoxLayout()
            col.addWidget(title)
            col.addWidget(view.widget, 1)
            row.addLayout(col)
        layout.addLayout(row, 10)

    def _build_side(self, layout):
        # hotspot list
        self.list_title = make_label(muted=True, bold=True)
        self.list = QtWidgets.QListWidget()
        self.list.setFocusPolicy(Qt.FocusPolicy.NoFocus)
        self.list.setSelectionMode(QtWidgets.QAbstractItemView.SelectionMode.NoSelection)
        self.list.setVerticalScrollBarPolicy(Qt.ScrollBarPolicy.ScrollBarAlwaysOff)
        self.list.setFont(QtGui.QFont("Consolas", 10))
        self.list.setFixedHeight(self.ROWS * 22 + 8)
        for _ in range(self.ROWS):
            self.list.addItem("")
        self.list.itemClicked.connect(lambda item: self._pick_row(self.list.row(item)))
        layout.addWidget(self.list_title)
        layout.addWidget(self.list)

        # data panel: names on the left, values on the right
        info = QtWidgets.QFrame()
        info.setProperty("panel", True)
        box = QtWidgets.QVBoxLayout(info)
        box.setContentsMargins(10, 8, 10, 8)
        self.info_title = make_label(bold=True)
        self.info_title_color = QtCore.QObject()          # just remembers the title's current colour
        self.info_keys, self.info_vals = make_label(size=9.5, muted=True), make_label(size=9.5)
        cols = QtWidgets.QHBoxLayout()
        for w in (self.info_keys, self.info_vals):
            w.setAlignment(Qt.AlignmentFlag.AlignTop | Qt.AlignmentFlag.AlignLeft)
            cols.addWidget(w)
        self.info_keys.setFixedWidth(120)
        cols.setStretch(1, 1)
        box.addWidget(self.info_title)
        box.addLayout(cols, 1)
        layout.addWidget(info, 5)

        # temperature chart
        self.chart_title = make_label(muted=True, size=8.5)
        self.chart = pg.PlotWidget(background=PANEL)
        pi = self.chart.getPlotItem()
        pi.showGrid(x=True, y=True, alpha=0.25)
        pi.setMenuEnabled(False)
        pi.hideButtons()
        pi.setMouseEnabled(False, False)
        pi.setXRange(-self.cfg.chart_seconds, 0, padding=0)
        for axis, text in (("bottom", "seconds ago"), ("left", self.scale.label)):
            pi.setLabel(axis, text, color=MUTED)
            pi.getAxis(axis).setTextPen(MUTED)
            pi.getAxis(axis).setPen(EDGE)
        self.chart_lines = []
        self.mean_line = pi.plot(pen=pg.mkPen(MUTED, width=1, style=Qt.PenStyle.DashLine))
        self.peak_line = pg.InfiniteLine(angle=0, pen=pg.mkPen(MUTED, width=1, style=Qt.PenStyle.DotLine))
        pi.addItem(self.peak_line)
        layout.addWidget(self.chart_title)
        layout.addWidget(self.chart, 4)

        # status line and buttons
        self.message_label = make_label()
        self.message_label.setStyleSheet("color: #ffd166;")
        layout.addWidget(self.message_label)
        buttons = QtWidgets.QHBoxLayout()
        for text, action in (("◀ Prev", lambda: self.cycle(-1)), ("Next ▶", lambda: self.cycle(1)),
                             ("Show all", lambda: self.select(None)), ("Save CSV", self.save_csv)):
            button = QtWidgets.QPushButton(text)
            button.setFocusPolicy(Qt.FocusPolicy.NoFocus)
            button.clicked.connect(lambda _=False, act=action: act())
            buttons.addWidget(button)
        layout.addLayout(buttons)

    def _bind_keys(self):
        keys = {"Q": self.window.close, "Escape": self.window.close, "S": self.save_csv,
                "A": lambda: self.select(None), "0": lambda: self.select(None)}
        keys.update({k: lambda: self.cycle(1) for k in ("N", "Right", "Down")})
        keys.update({k: lambda: self.cycle(-1) for k in ("P", "Left", "Up")})
        keys.update({str(i + 1): lambda i=i: self._pick_row(i) for i in range(self.ROWS)})
        self.shortcuts = []
        for key, action in keys.items():
            shortcut = QShortcut(QtGui.QKeySequence(key), self.window)
            shortcut.setContext(Qt.ShortcutContext.ApplicationShortcut)
            shortcut.activated.connect(action)
            self.shortcuts.append(shortcut)                 # shortcuts stop working if not kept

    # ---- picking a hotspot ----

    @property
    def closed(self):
        return self.window.closed

    def pump(self):
        """Let Qt paint and handle clicks/keys."""
        self.qapp.processEvents()

    def select(self, tid):
        if tid != self.selected:
            self.selected = tid
            self.history.reset()                        # 3D views restart with only this hotspot
            self.soft_key, self.last_panels, self.panel_step = None, -np.inf, 0
            self.flash(f"Tracking hotspot #{tid} only" if tid is not None else "Showing all hotspots")

    def cycle(self, step):
        ids = [t.id for t in self.tracker.listed()]
        if not ids:
            self.flash("No hotspots to pick")
        elif self.selected in ids:
            self.select(ids[(ids.index(self.selected) + step) % len(ids)])
        else:
            self.select(ids[0] if step > 0 else ids[-1])

    def _pick_row(self, row):
        if 0 <= row < len(self.row_ids):
            self.select(self.row_ids[row])

    def _on_map_click(self, event):
        vb = self.live_plot.vb
        if event.button() != Qt.MouseButton.LeftButton or not vb.sceneBoundingRect().contains(event.scenePos()):
            return
        p = vb.mapSceneToView(event.scenePos())
        pad = 3
        hits = [t for t in self.drawn
                if t.det.box[0] - pad <= p.x() <= t.det.box[0] + t.det.box[2] + pad
                and t.det.box[1] - pad <= p.y() <= t.det.box[1] + t.det.box[3] + pad]
        if hits:
            self.select(min(hits, key=lambda t: t.det.area).id)
        else:
            self.flash("No hotspot there - click inside a box")

    def flash(self, text, seconds=3.0):
        self.message, self.message_until = text, time.monotonic() + seconds

    def save_csv(self):
        t = self.tracker.get(self.selected)
        if t is None:
            return self.flash("Pick a hotspot first, then save its data")
        name = f"hotspot_{t.id}_{time.strftime('%Y%m%d_%H%M%S')}.csv"
        data = np.array(t.log, np.float64)
        data[:, 0] -= t.first_seen
        np.savetxt(name, data, delimiter=",", fmt="%.3f", comments="",
                   header="time_s,temp_max,temp_mean,center_x,center_y,box_x,box_y,box_w,box_h")
        print(f"Saved {len(data)} samples of hotspot #{t.id} to {name}")
        self.flash(f"Saved {name}")

    # ---- every frame ----

    def focus_values(self, values):
        """The frame with everything except the picked hotspot flattened to the scene median."""
        if self.selected is None:
            return values
        ambient, t = self.detector.ambient, self.tracker.get(self.selected)
        if t is None:
            return np.full_like(values, ambient)
        if self.soft_key != (t.id, t.hits):             # rebuild the soft mask only when it changed
            grown = cv.dilate(t.det.mask.astype(np.uint8), self.kernel).astype(np.float32)
            self.soft, self.soft_key = cv.GaussianBlur(grown, (0, 0), 1.5), (t.id, t.hits)
        return (ambient + (values - ambient) * self.soft).astype(np.float32)

    def draw(self, values, focused, now, frac, draw_3d=True, draw_panel=True):
        """Update the window for this frame (Qt paints it on the next pump()).

        frac: how far (0..1) the 3D views are into the current row.
        draw_3d / draw_panel: lets run() spread the expensive parts over different frames.
        """
        self.frames += 1
        if now - self.fps_t0 >= 1.0:
            self.fps, self.frames, self.fps_t0 = self.frames / (now - self.fps_t0), 0, now

        s, size = self.scale, (2 * self.width, 2 * self.height)
        for image, data in ((self.live_img, values), (self.focus_img, focused)):
            image.set_pixels(cv.resize(LUT_BGRA[colour_index(data, s.low, s.span)], size, interpolation=cv.INTER_LINEAR))
        listed, picked = self.tracker.listed(), self.tracker.get(self.selected)
        self._draw_maps(listed, picked)
        set_text(self.message_label, self.message if time.monotonic() < self.message_until else "")

        if draw_3d and self.history.rows:
            ease = min((now - self.last_3d) / self.cfg.update_seconds, 4.0)
            self.last_3d = now
            grid = cv.GaussianBlur(self.history.with_live_row(), (0, 0), sigmaX=self.cfg.blur_x, sigmaY=self.cfg.blur_t)
            for view in self.views:
                view.draw(grid, frac, self.history.serial, ease)

        # Side panels: one piece per frame, every panel_seconds.
        if draw_panel and (self.panel_step or now - self.last_panels >= self.cfg.panel_seconds):
            if self.panel_step == 0:
                self.last_panels = now
            (self._draw_list, self._draw_info, self._draw_chart, self._draw_titles)[self.panel_step](listed, picked, now)
            self.panel_step = (self.panel_step + 1) % 4

    def _draw_maps(self, listed, picked):
        shown = self.drawn = [t for t in listed if t.missed <= 2]
        for i, t in enumerate(shown):
            if picked is None:
                self._place_box(self.live_plot, self.live_boxes, i, t, 1.4, False, 1.0)
            elif t is picked:
                self._place_box(self.live_plot, self.live_boxes, i, t, 2.6, False, 1.0)
            else:
                self._place_box(self.live_plot, self.live_boxes, i, t, 1.0, True, 0.45)
        self._hide_boxes(self.live_boxes, len(shown))
        n = len(shown)
        self._title(self.live_plot, f"LIVE MAP  ·  {n} hotspot{'s' if n != 1 else ''}  ·  click a box to track it"
                                    f"  ·  {self.fps:.0f} fps")

        if picked is not None:
            self._place_box(self.focus_plot, self.focus_boxes, 0, picked, 2.6, picked.missed != 0, 1.0)
            peak = [picked.det.peak_xy[0]], [picked.det.peak_xy[1]]
            self.focus_note.setVisible(False)
            title = f"TRACKED ONLY  ·  hotspot #{picked.id}"
        else:
            self._hide_boxes(self.focus_boxes)
            peak = [], []
            gone = self.selected is not None
            set_text(self.focus_note, f"Hotspot #{self.selected} is gone\npick another one" if gone
                     else "Nothing picked: showing every hotspot\nclick a box on the live map")
            self.focus_note.setVisible(gone or not shown)
            title = "TRACKED ONLY  ·  " + (f"#{self.selected} (gone)" if gone else "all hotspots")
        for marker in (self.live_peak, self.focus_peak):
            marker.setData(*peak)
        self._title(self.focus_plot, title)

    def _place_box(self, plot, pool, i, t, width, dashed, alpha):
        """Box + label number i of `pool` around hotspot t (boxes are reused, never deleted)."""
        while len(pool) <= i:
            rect = QtWidgets.QGraphicsRectItem()
            rect.setBrush(QtGui.QBrush(Qt.BrushStyle.NoBrush))
            label = pg.TextItem("", fill=pg.mkBrush(0, 0, 0, 150))
            font = QtGui.QFont("Segoe UI", 8)
            font.setBold(True)
            label.setFont(font)
            for z, item in ((5, rect), (6, label)):
                item.setZValue(z)
                plot.addItem(item)
            pool.append([rect, label, None])
        rect, label, style = pool[i]
        if style != (t.id, width, dashed, alpha):       # restyle only when the look changes
            color = spot_color(t.id)
            rect.setPen(pg.mkPen(color, width=width, style=Qt.PenStyle.DashLine if dashed else Qt.PenStyle.SolidLine))
            label.setColor(color)
            rect.setOpacity(alpha)
            label.setOpacity(alpha)
            pool[i][2] = (t.id, width, dashed, alpha)
        x, y, w, h = t.det.box
        rect.setRect(x - 0.5, y - 0.5, w, h)
        above, right = y > 14, x > self.width - 60      # keep the label inside the image
        label.setAnchor((1 if right else 0, 1 if above else 0))
        label.setPos(x + w - 0.5 if right else x - 0.5, y - 1.5 if above else y + h + 0.5)
        set_text(label, f"#{t.id}  {t.det.peak:.1f} {self.scale.label}")
        rect.setVisible(True)
        label.setVisible(True)

    @staticmethod
    def _hide_boxes(pool, start=0):
        for rect, label, _ in pool[start:]:
            rect.setVisible(False)
            label.setVisible(False)

    @staticmethod
    def _title(plot, text):
        set_text(plot, text, lambda s: plot.setTitle(s, color=TEXT, size="10pt", justify="left"))

    # ---- side panels (a few times a second) ----

    def _draw_list(self, listed, picked, now):
        rows = listed[:self.ROWS]
        self.row_ids = [t.id for t in rows]
        more = len(listed) - len(rows)
        set_text(self.list_title, f"HOTSPOTS  ·  {len(listed)}" + (f"  (+{more} not shown)" if more else "")
                 + ("" if rows else "   ·   none right now"))
        for i in range(self.ROWS):
            item = self.list.item(i)
            item.setHidden(i >= len(rows))
            if i >= len(rows):
                continue
            t = rows[i]
            color = QtGui.QColor(spot_color(t.id))
            color.setAlphaF(0.45 if picked is not None and t is not picked else 1.0)
            status = "live" if t.missed == 0 else f"missing {now - t.last_seen:.0f}s"
            item.setText(f"■ #{t.id:<3d}{t.det.peak:6.1f} {self.scale.label}   peak {t.high:5.1f}   {status}")
            item.setForeground(color)
            item.setBackground(QtGui.QColor("#22314a") if t is picked else QtGui.QColor(0, 0, 0, 0))

    def _draw_info(self, listed, picked, now):
        u, s = self.scale.label, self.scale
        if picked is not None:
            t, d = picked, picked.det
            ago = now - t.last_seen
            status = ("tracking" if t.missed == 0 else f"searching... {ago:.1f} s"
                      if t.missed <= self.cfg.forget_frames else f"lost, last seen {ago:.0f} s ago")
            cx, cy = t.center
            rows = [("Hotspot", f"#{t.id}"), ("Status", status),
                    ("Temp now", f"{d.peak:.1f} {u}" + ("" if t.missed == 0 else "  (last seen)")),
                    ("Region mean", f"{d.mean:.1f} {u}"), ("Recorded peak", f"{t.high:.1f} {u}"),
                    ("Recorded low", f"{t.low:.1f} {u}"), ("Recorded avg", f"{t.average:.1f} {u}"),
                    ("Recorded for", f"{t.last_seen - t.first_seen:.1f} s  ({t.hits} samples)"),
                    ("Position", f"x {cx:.0f}, y {cy:.0f} px"),
                    ("Box size", f"{d.box[2]} x {d.box[3]} px  ({d.area} px)"),
                    ("Distance moved", f"{t.distance:.0f} px"),
                    ("From start", f"{np.hypot(cx - t.start[0], cy - t.start[1]):.0f} px"),
                    ("Speed", f"{t.speed():.1f} px/s")]
            title, color = f"PICKED HOTSPOT  #{t.id}", spot_color(t.id)
        else:
            live = [t for t in listed if t.missed == 0]
            hot = max(live, key=lambda t: t.det.peak, default=None)
            rows = [("Tracking", "all hotspots" if self.selected is None else f"#{self.selected} (gone)"),
                    ("Hotspots now", str(len(live))),
                    ("Hottest", f"#{hot.id}   {hot.det.peak:.1f} {u}" if hot else "-"),
                    ("Scene median", f"{self.detector.ambient:.1f} {u}"),
                    ("Hotspot above", f"{self.detector.threshold:.1f} {u}"),
                    ("Colour scale", f"{s.low:.1f} to {s.high:.1f} {u}"),
                    ("Frame rate", f"{self.fps:.1f} fps"), ("", ""),
                    ("Tip", "click a box or a list row"), ("", "to track a single hotspot")]
            title, color = "OVERVIEW", MUTED
        set_text(self.info_title, title)
        set_text(self.info_title_color, color, lambda c: self.info_title.setStyleSheet(f"color: {c};"))
        set_text(self.info_keys, "\n".join(k for k, _ in rows))
        set_text(self.info_vals, "\n".join(v for _, v in rows))

    def _draw_chart(self, listed, picked, now):
        pi = self.chart.getPlotItem()
        tracks = [picked] if picked is not None else [t for t in listed if t.missed <= self.cfg.forget_frames][:8]
        while len(self.chart_lines) < len(tracks):
            self.chart_lines.append(pi.plot())
        lo, hi = np.inf, -np.inf
        for line, t in zip(self.chart_lines, tracks):
            data = np.array([e[:3] for e in t.recent(now - self.cfg.chart_seconds)]).reshape(-1, 3)
            line.setData(data[:, 0] - now, data[:, 1])
            line.setPen(pg.mkPen(spot_color(t.id), width=2.0 if picked is not None else 1.3))
            line.setVisible(True)
            if len(data):
                lo, hi = min(lo, data[:, 1:].min()), max(hi, data[:, 1].max())
            if t is picked:
                self.mean_line.setData(data[:, 0] - now, data[:, 2])
                self.peak_line.setValue(t.high)
                hi = max(hi, t.high)
        for line in self.chart_lines[len(tracks):]:
            line.setVisible(False)
        self.mean_line.setVisible(picked is not None)
        self.peak_line.setVisible(picked is not None)
        if not np.isfinite(lo):
            lo, hi = self.scale.low, self.scale.high
        pad = max(0.5, 0.08 * (hi - lo))
        pi.setYRange(lo - pad, hi + pad, padding=0)
        set_text(self.chart_title, f"TEMPERATURE  #{picked.id}   (solid max, dashed mean, dotted recorded peak)"
                 if picked is not None else "TEMPERATURE  ·  all hotspots (max)")

    def _draw_titles(self, listed, picked, now):
        self.cbar.setLevels((self.scale.low, self.scale.high))
        what = f"hotspot #{self.selected}" if self.selected is not None else "all hotspots"
        for name, view, title in zip(("3D map", "Wisps"), self.views, self.view_titles):
            set_text(title, f"{name}  ·  {what}  ·  {view.range_text}")


# ==============================================================================================
# 7. Main loop
# ==============================================================================================

def run(cfg=Config()):
    camera = SimulatedCamera(cfg) if cfg.simulate else Camera(cfg)
    pacer = FramePacer(cfg.sim_fps) if cfg.simulate else None
    detector, tracker = HotspotDetector(cfg), Tracker(cfg)
    if pacer:
        how = f"locked to the {1 / pacer.refresh:.0f} Hz screen" if pacer.refresh else "timer"
        print(f"SIMULATION at {pacer.fps:.0f} fps ({how}). Set Config.simulate=False for the real camera.")
    else:
        print("Reading temperatures (°C)." if camera.raw else "Reading brightness only: this camera sends processed frames.")

    def next_frame():
        """Read and analyse a frame. Runs on a helper thread, overlapping the previous frame's painting."""
        values = camera.read()
        if values is None:
            return None
        camera.scale.update(values)
        return values, detector.detect(values, camera.scale), time.monotonic()

    # Above ~40 fps the 3D views redraw every other frame and the side panels use the frames in
    # between, so no single frame does everything and misses its screen refresh.
    split = pacer is not None and pacer.fps > 40
    worker = ThreadPoolExecutor(max_workers=1)
    job = worker.submit(next_frame)
    app = history = None
    last_row, frame_no = -np.inf, 0
    try:
        while True:
            while app is not None and not job.done() and not app.closed:   # real camera: stay responsive
                app.pump()
                time.sleep(0.002)
            frame = job.result()
            if frame is None:
                break
            job = worker.submit(next_frame)             # start the next frame right away
            values, found, now = frame
            tracker.update(found, now, keep=app.selected if app else None)

            if app is None:
                history = History(values.shape[1], cfg)
                app = ThermalApp(cfg, camera.scale, history, tracker, detector, values.shape, cfg.simulate)

            focused = app.focus_values(values)
            history.add_frame(focused)
            if now - last_row >= cfg.update_seconds:
                last_row = max(last_row + cfg.update_seconds, now - 0.5 * cfg.update_seconds)  # steady rhythm
                history.commit()
            frac = min(max((now - last_row) / cfg.update_seconds, 0.0), 1.0)

            frame_no += 1
            draw_3d = not split or frame_no % 2 == 0
            app.draw(values, focused, now, frac, draw_3d=draw_3d, draw_panel=not split or not draw_3d)
            if pacer:
                pacer.wait()                            # paint right after a screen refresh
            app.pump()
            if app.closed:
                break
    finally:
        worker.shutdown(wait=True, cancel_futures=True)
        camera.release()
        if pacer:
            print(pacer.report())
        if app is not None:
            app.window.close()
            app.window.deleteLater()
            app.pump()


if __name__ == "__main__":
    run()

SIMULATION at 71 fps (locked to the 143 Hz screen). Set Config.simulate=False for the real camera.
This computer can't keep up: switching to a steady 48 fps.
Ran 3365 frames in 68 s (49 fps), 98% on time. Final rate 48 fps.
